# CIFAR-10 CNN EAUT ExperimentNotebook này dùng để chạy lại thực nghiệm từ GitHub trên Google Colab.

## 1. Kiểm tra GPU

In [1]:
!nvidia-smi || true

Mon Jun 22 16:00:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone project từ GitHub
Sau khi đưa project lên GitHub, sửa `REPO_URL` thành link repository.

In [2]:
REPO_URL = "https://github.com/hoangtnedu/cifar10-cnn-eaut.git"
PROJECT_DIR = "cifar10-cnn-eaut"

import os, shutil
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
!git clone $REPO_URL

Cloning into 'cifar10-cnn-eaut'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 51 (delta 16), reused 29 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 30.37 KiB | 10.12 MiB/s, done.
Resolving deltas: 100% (16/16), done.


In [3]:
%cd cifar10-cnn-eaut

!pip install -q -r requirements.txt

/content/cifar10-cnn-eaut


## 2.1. Gắn Google Drive để resume bền hơn

Dùng để checkpoint không mất khi Colab reset runtime, hãy mount Google Drive và dùng config `cifar10_5seeds_colab_drive.yaml`.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

MAIN_CONFIG = 'configs/cifar10_5seeds_colab_drive.yaml'  # lưu checkpoint/kết quả vào Google Drive
# MAIN_CONFIG = 'configs/cifar10_5seeds.yaml'  # dùng nếu không muốn lưu Drive

Mounted at /content/drive


## 3. Test nhanh bằng config debug



Chạy 1 model, 1 seed, 2 epoch để kiểm tra code trước.

In [ ]:
!python run_experiments.py --config configs/debug.yaml

!python aggregate_results.py --config configs/debug.yaml

## 4. Chạy thực nghiệm chính 5 seed



Lệnh này có thể chạy lâu. Nếu Colab ngắt, chạy lại cell này; project sẽ tự resume từ `checkpoint_last.pt`.

In [ ]:
!python run_experiments.py --config $MAIN_CONFIG

## 5. Tổng hợp kết quả mean ± std

In [ ]:
!python aggregate_results.py --config $MAIN_CONFIG
!cat /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/aggregate/summary_markdown.md || cat outputs/cifar10_5seeds/aggregate/summary_markdown.md

## 6. Sinh hình và bảng cho bài báo

Cell này đọc kết quả đã huấn luyện và sinh các bảng `.csv`, `.md` cùng các hình `.png` phục vụ đưa vào bài báo. Mặc định script sẽ tạo cả ma trận nhầm lẫn cho seed tốt nhất của từng mô hình. Nếu chỉ muốn sinh nhanh bảng và biểu đồ tổng quan, bỏ comment dòng có `--skip-confusion-matrix`.

In [ ]:
!python make_figures.py --config $MAIN_CONFIG
# Chạy nhanh, không tạo confusion matrix:
# !python make_figures.py --config $MAIN_CONFIG --skip-confusion-matrix

In [5]:
!echo "=== Figures ==="
!find /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures -maxdepth 1 -type f 2>/dev/null | sort || find outputs/cifar10_5seeds/figures -maxdepth 1 -type f 2>/dev/null | sort

!echo "=== Paper tables ==="
!find /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/paper_tables -maxdepth 1 -type f 2>/dev/null | sort || find outputs/cifar10_5seeds/paper_tables -maxdepth 1 -type f 2>/dev/null | sort

!echo "=== Main results table ==="
!cat /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/paper_tables/table_1_main_results_for_paper.md || cat outputs/cifar10_5seeds/paper_tables/table_1_main_results_for_paper.md

=== Figures ===
/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures/fig_01_accuracy_f1_bar.png
/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures/fig_02_params_bar.png
/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures/fig_03_flops_bar.png
/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures/fig_04_latency_bar.png
/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures/fig_05_accuracy_vs_params.png
/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures/fig_06_accuracy_vs_flops.png
/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures/fig_07_accuracy_vs_latency.png
/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures/fig_08_pareto_accuracy_latency.png
/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures/fig_09_validation_accuracy_curves_all_models.png
/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures/fig_confusion_matrix_normalized_EfficientNetB0C

## 6.1. Hiển thị nhanh một số hình chính

In [ ]:
from pathlib import Path
from IPython.display import Image, display

base_dir = Path('/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds')
if not base_dir.exists():
    base_dir = Path('outputs/cifar10_5seeds')

fig_dir = base_dir / 'figures'
for fig_name in [
    'fig_01_accuracy_f1_bar.png',
    'fig_05_accuracy_vs_params.png',
    'fig_06_accuracy_vs_flops.png',
    'fig_07_accuracy_vs_latency.png',
    'fig_08_pareto_accuracy_latency.png',
    'fig_09_validation_accuracy_curves.png',
]:
    fig_path = fig_dir / fig_name
    if fig_path.exists():
        print(fig_name)
        display(Image(filename=str(fig_path)))
    else:
        print(f'Missing: {fig_path}')

## 7. Nén kết quả để tải về hoặc lưu Google Drive

In [ ]:
!zip -r cifar10_5seeds_outputs.zip /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds outputs/cifar10_5seeds 2>/dev/null > /dev/null
from google.colab import files
files.download('cifar10_5seeds_outputs.zip')

In [7]:
from pathlib import Path
import shutil

# Đường dẫn đúng theo Google Drive của anh:
src = Path("/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds")

# Thư mục tạm để gom file nhẹ
dst = Path("/content/cifar10_results_for_paper")

folders = [
    "aggregate",
    "paper_tables",
    "figures",
    "SimpleCNN",
    "VGG11TinyBN",
    "ResNet18CIFAR",
    "MobileNetV2CIFAR",
    "EfficientNetB0CIFAR",
]

exts = {".csv", ".json", ".xlsx", ".png", ".jpg", ".jpeg", ".txt", ".log", ".md"}

print("Kiểm tra thư mục nguồn:", src)
print("Tồn tại không?", src.exists())

if not src.exists():
    raise FileNotFoundError(f"Không tìm thấy thư mục nguồn: {src}")

if dst.exists():
    shutil.rmtree(dst)
dst.mkdir(parents=True, exist_ok=True)

count = 0

for folder in folders:
    base = src / folder
    print("Kiểm tra:", base, "=>", base.exists())

    if not base.exists():
        continue

    for p in base.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            rel = p.relative_to(src)
            out = dst / rel
            out.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, out)
            count += 1

print("Số file đã copy:", count)

zip_path = shutil.make_archive("/content/cifar10_results_for_paper", "zip", dst)

print("Đã tạo file:", zip_path)
print("Dung lượng zip:", Path(zip_path).stat().st_size / 1024 / 1024, "MB")

Kiểm tra thư mục nguồn: /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds
Tồn tại không? True
Kiểm tra: /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/aggregate => True
Kiểm tra: /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/paper_tables => True
Kiểm tra: /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures => True
Kiểm tra: /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/SimpleCNN => True
Kiểm tra: /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/VGG11TinyBN => True
Kiểm tra: /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/ResNet18CIFAR => True
Kiểm tra: /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/MobileNetV2CIFAR => True
Kiểm tra: /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/EfficientNetB0CIFAR => True
Số file đã copy: 91
Đã tạo file: /content/cifar10_results_for_paper.zip
Dung lượng zip: 4.268486022949219 MB
